# RetailMind AI — Day 6: a CNN that classifies product images

**Scenario.** RetailMind AI is built one increment per day in the same repo. Days 1-5 built the churn pipeline, compared classical models and a Keras ANN (the classical model stayed in production). Day 6 starts the **computer-vision** side of the product.

**Objective.** Add a vision module that classifies product images automatically. Auto-tagging the catalog cuts manual cataloguing effort and powers visual search later.

**How it connects to earlier days**
- Same repo, same style: configuration in `src/config.py`, a `train_dayN.py` script and a small service file.
- Same Keras habits as Day 5: early stopping on validation loss, choosing the model by validation (never test) and a fixed random seed.
- Day 5 showed that a neural network is *not* the best tool for a table of customers. Day 6 uses it where it should shine: images.

**Business value.** Every product photo gets a category tag without a person typing it in, which cuts cataloguing effort and gives visual search a first building block.

**Data note.** The prototype uses Fashion-MNIST (built into Keras). The real *Fashion Product Images* (Kaggle) are not downloaded here, so the real-photo step is written and tested on synthetic pictures but has **not** been run on real data.

In [1]:
# Work from the project root so the `src` package can be imported
import os, sys
ROOT = os.path.abspath("..") if os.path.basename(os.getcwd()) == "notebooks" else os.getcwd()
os.chdir(ROOT)
sys.path.insert(0, ROOT)
print("project folder:", os.path.basename(ROOT))

project folder: day3-4


## 1. Look at the pictures
Fashion-MNIST: 60,000 training and 10,000 test pictures, 28x28 pixels, grey, 10 categories.

In [2]:
from src.train_day6 import load_fashion_mnist, FASHION_CLASSES
x_train, y_train, x_test, y_test = load_fashion_mnist()
print("training pictures:", x_train.shape, "| test pictures:", x_test.shape)
print("categories:", FASHION_CLASSES)

# show one picture of each category as text (no plotting library needed)
shades = " .:-=+*#%@"
for category in (1, 9):
    index = int((y_train == category).nonzero()[0][0])
    print("\n" + FASHION_CLASSES[category])
    for row in x_train[index, ::2, ::1, 0]:
        print("".join(shades[int(v * 9)] for v in row))

training pictures: (60000, 28, 28, 1) | test pictures: (10000, 28, 28, 1)
categories: ['T-shirt/top', 'Trouser', 'Pullover', 'Dress', 'Coat', 'Sandal', 'Shirt', 'Sneaker', 'Bag', 'Ankle boot']

Trouser
         .+==-==-++*        
         =+****##*+%.       
         *#########%-       
         #++**#%##*#*       
        .*=+*# %*++**       
        .*=++# **+***       
        .+=+*# =#=+**       
        .*=+** .#++**       
        :*=+*=  #++*+       
        .#=*#.  **+*+       
        .#+*#   =**#+       
        :#+#*   -**#+       
        :*+#+   .**#+       
        -**#=   .**#+       

Ankle boot
                            
                            
              .==:.         
              +%#*-++-: :=: 
             #%%%%####+==*% 
             *%##**######%# 
            -%####*#####%=+.
            %#########%##@: 
     .:-*%#######%#%**%%%%# 
 #%#########%:+@%#*+*####%# 
:###*####**#*#%*%%%#######%:
 =#*****#######********++*#-
    :##%%%%%%##*******++-:  
   

## 2. Train and compare: dense network vs CNN vs CNN with augmentation
`main()` does the whole increment:
1. trains three designs on the same pictures: a **dense** network (flattened pixels), a **CNN** (conv → pool → conv → pool → dropout → dense) and the same CNN with **augmentation** (random flips, turns and zooms, applied only while training)
2. stops each one with **early stopping** on validation loss and picks the design by validation loss, never by the test pictures
3. prints the **confusion matrix** of the chosen model
4. saves the catalog classifier (`models/catalog_cnn.keras`)
5. runs the same idea on real product photos *if* they are on disk (otherwise explains how to get them)

In [3]:
from src.train_day6 import main
main()

Step 1: prototype on Fashion-MNIST (28x28 pictures, 10 categories)
  60000 training pictures, 10000 test pictures


  Dense network (flattened pixels): 101770 weights, 21 rounds, best validation loss 0.3216


  CNN, no augmentation: 121930 weights, 28 rounds, best validation loss 0.2175


  CNN with augmentation: 121930 weights, 40 rounds, best validation loss 0.2541
  lower validation loss wins -> CNN, no augmentation

Step 2: test pictures
  Dense network (flattened pixels)   test accuracy 0.8824


  CNN, no augmentation               test accuracy 0.9163


  CNN with augmentation              test accuracy 0.9027

  Confusion matrix for CNN, no augmentation (rows = real, columns = guessed):
               T-shi Trous Pullo Dress  Coat Sanda Shirt Sneak   Bag Ankle
  T-shirt/top    878     0    16    14     4     1    81     0     6     0
  Trouser          3   979     0    14     1     0     2     0     1     0
  Pullover        16     0   901     6    29     0    48     0     0     0
  Dress           14     3     8   924    21     0    30     0     0     0
  Coat             3     0    66    26   855     0    48     0     2     0
  Sandal           0     0     0     0     0   988     0     6     0     6
  Shirt          123     0    58    26    58     0   731     0     4     0
  Sneaker          0     0     0     0     0    13     0   963     0    24
  Bag              2     0     4     2     1     5     3     1   981     1
  Ankle boot       0     0     0     0     0     7     0    29     1   963
  Weakest categories (recall): [('Shir

## 3. The comparison table

In [4]:
import json
import pandas as pd

with open("models/day6_results.json") as f:
    day6 = json.load(f)

table = pd.DataFrame({
    name: {"weights": info["training"]["parameters"],
           "rounds trained": info["training"]["rounds_trained"],
           "best round": info["training"]["best_round"],
           "best validation loss": info["training"]["best_validation_loss"],
           "TEST accuracy": info["test_accuracy"]}
    for name, info in day6["models"].items()}).T
display(table)
print("Chosen (lowest validation loss):", day6["chosen_model"])
print("CNN minus dense accuracy:", day6["cnn_minus_dense_accuracy"])
print("Augmentation minus no augmentation:", day6["augmentation_minus_none_accuracy"])

,weights,rounds trained,best round,best validation loss,TEST accuracy
Dense network (flattened pixels),101770.0,21.0,16.0,0.3216,0.8824
"CNN, no augmentation",121930.0,28.0,23.0,0.2175,0.9163
CNN with augmentation,121930.0,40.0,36.0,0.2541,0.9027


Chosen (lowest validation loss): CNN, no augmentation
CNN minus dense accuracy: 0.0339
Augmentation minus no augmentation: -0.0136


## 4. Which categories does the model confuse?
Recall per category (of all real pictures of this category, the share we got right), and the biggest mix-ups.

In [5]:
chosen = day6["chosen_model_test"]
display(pd.Series(chosen["per_class_recall"], name="recall").sort_values().to_frame())
display(pd.DataFrame(chosen["top_mix_ups"]))

,recall
Shirt,0.731
Coat,0.855
T-shirt/top,0.878
Pullover,0.901
Dress,0.924
Sneaker,0.963
Ankle boot,0.963
Trouser,0.979
Bag,0.981
Sandal,0.988


,times,real,guessed_as
0,123,Shirt,T-shirt/top
1,81,T-shirt/top,Shirt
2,66,Coat,Pullover
3,58,Shirt,Pullover
4,58,Shirt,Coat


## 5. Use the saved classifier
`src/catalog_service.py` loads the saved model and tags pictures. Here it tags 8 test pictures and we compare with the real category.

In [6]:
from src.catalog_service import classify_images

samples = (x_test[:8, :, :, 0] * 255).astype("uint8")
for answer, real in zip(classify_images(samples), y_test[:8]):
    print(f"picture {answer['picture']}: guessed {answer['category']:<12} (confidence {answer['confidence']}) | real: {FASHION_CLASSES[int(real)]}")

picture 1: guessed Ankle boot   (confidence 1.0) | real: Ankle boot
picture 2: guessed Pullover     (confidence 1.0) | real: Pullover
picture 3: guessed Trouser      (confidence 1.0) | real: Trouser
picture 4: guessed Trouser      (confidence 1.0) | real: Trouser
picture 5: guessed Shirt        (confidence 0.9158) | real: Shirt
picture 6: guessed Trouser      (confidence 1.0) | real: Trouser
picture 7: guessed Coat         (confidence 0.979) | real: Coat
picture 8: guessed Shirt        (confidence 0.9691) | real: Shirt


## 6. Conclusion

**CNN vs dense network: the CNN wins.** Test accuracy 0.9163 vs 0.8824 (+0.034), with a similar number of weights (121,930 vs 101,770). A dense network flattens the picture and loses the idea of "neighbouring pixels", while convolution filters look for edges and shapes anywhere in the picture. This is where a neural network earns its place (compare Day 5, where it did not beat the classical model on a table of customers).

**Augmentation did not help here.** The augmented CNN scored 0.9027, below the plain CNN's 0.9163, and it used all 40 allowed rounds, so it was still creeping up. Fashion-MNIST pictures are already centred and the same size, so random flips, turns and zooms mostly add noise. Augmentation matters more for real, messy product photos, which is the reason it is in the real-photo pipeline.

**Where the model is weak.** Shirt is the weakest category (recall 0.731). It is most often called T-shirt/top (123 times), Pullover (58) and Coat (58). These are tops that look alike at 28x28, so the confusion is understandable, and it suggests that higher-resolution real photos (Day 7: transfer learning) should help most there.

**Decision.** `models/catalog_cnn.keras` (the plain CNN, chosen by validation loss) is the first computer-vision module of RetailMind, and `src/catalog_service.py` tags pictures with it.

**Open blockers / notes for doubt-clearing**
- The real *Fashion Product Images* data (Kaggle, needs an account to download) was not available, so the real-photo step has only been tested on synthetic pictures, not on real data. No real-photo accuracy is claimed.
- Each model was trained once with a fixed seed. Gaps of about 0.01 (for example between the CNN and the augmented CNN) are within what another seed could change.
- Days 3-6 are on stacked, still-unmerged branches (`day3-4-tuning-and-ensembles` -> `day5-keras-ann` -> `day6-cnn-catalog`).